In [1]:
import torch
import cv2
import numpy as np

import os
from extraction.images import model_loader
from extraction.video_processing import extract_video_features_compressed_ms

import time

In [2]:
adult_dir = ['data/images/tikharm/train/Adult Content']
safe_dir = ['data/images/tikharm/train/Safe']

In [3]:
if torch.backends.mps.is_available():
    device = "mps"
elif torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

In [4]:
# 1. Device Guard Setup
device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"⏳ Loading transformer weights into active hardware storage space...")

# 2. Initialize weights ONCE at the top level of the cell
global_model, global_processor, active_code = model_loader(model_code='clip', device=device)
print(f"✅ Transformer successfully cached on: {device.upper()}\n")

⏳ Loading transformer weights into active hardware storage space...
✅ Transformer successfully cached on: MPS



In [5]:
adult_data = []

# ─── PROCESS VIOLENT DATASET TRACK ───
for video_dir in adult_dir:
    if not os.path.isdir(video_dir): continue
        
    for video_file in os.listdir(video_dir):
        # Skip system garbage metadata files like .DS_Store
        if video_file.startswith('.'): continue 

        video_path = os.path.join(video_dir, video_file)
        start_timer = time.time()
        
        # 🔥 FIXED: Passing global_model and global_processor variables smoothly instead of string codes
        vector = extract_video_features_compressed_ms(
            video_path=video_path,
            model=global_model,
            processor=global_processor,
            model_code=active_code,
            target_frames=18
        )
        
        adult_data.append(vector)
        execution_speed = time.time() - start_timer

In [7]:
safe_data = []

# ─── PROCESS VIOLENT DATASET TRACK ───
for video_dir in safe_dir:
    if not os.path.isdir(video_dir): continue
    
    for video_file in os.listdir(video_dir)[:150]:
        # Skip system garbage metadata files like .DS_Store
        if video_file.startswith('.'): continue 

        video_path = os.path.join(video_dir, video_file)      
        start_timer = time.time()
        
        # 🔥 FIXED: Passing global_model and global_processor variables smoothly instead of string codes
        vector = extract_video_features_compressed_ms(
            video_path=video_path,
            model=global_model,
            processor=global_processor,
            model_code=active_code,
            target_frames=18
        )
        
        safe_data.append(vector)
        execution_speed = time.time() - start_timer

In [8]:
from sklearn.model_selection import train_test_split
from training.kfold_train import stratified_kfold_train_val

from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

from sklearn.metrics import confusion_matrix, classification_report

In [14]:
adult_labels = np.ones(len(adult_data))[:600]
#harmful_labels = np.ones(len(harmful_data))[:600]
safe_labels = np.zeros(len(safe_data))

adult_data = np.array(adult_data)[:600]
#harmful_data = np.array(harmful_data)[:600]
safe_data = np.array(safe_data)

In [15]:
x_adult = np.concat([adult_data, safe_data])
y_adult = np.concat([adult_labels, safe_labels])

#x_harmful = np.concat([harmful_data, safe_data])
#y_harmful = np.concat([harmful_labels, safe_labels])

In [16]:
x_train, x_test, y_train, y_test = train_test_split(x_adult, y_adult,
                                                    stratify=y_adult,
                                                    test_size=0.2)

In [17]:
log_reg = LogisticRegression(
    penalty='l2',          # Use L2 (Ridge) regularization
    C=0.5,                 # Slightly stronger regularization than default
    solver='liblinear',        # Standard efficient solver
    max_iter=1000,         # Increased to guarantee optimization convergence
    random_state=42,
    class_weight='balanced'
)

stratified_kfold_train_val(5,
                           0.26,
                           log_reg,
                           x_train,
                           y_train)

Starting 5-Fold Stratified Cross-Validation...

sss
--- Performance Analysis at Threshold (0.35) ---
Custom Precision Score: 0.9792 (When flagged positive, accuracy is 97.92%)
Custom Recall Score:    0.9792 (Captured 97.92% of all true positive cases)
sss
--- Performance Analysis at Threshold (0.35) ---
Custom Precision Score: 0.9792 (When flagged positive, accuracy is 97.92%)
Custom Recall Score:    0.9792 (Captured 97.92% of all true positive cases)
sss
--- Performance Analysis at Threshold (0.35) ---
Custom Precision Score: 0.9500 (When flagged positive, accuracy is 95.00%)
Custom Recall Score:    0.9896 (Captured 98.96% of all true positive cases)
sss
--- Performance Analysis at Threshold (0.35) ---
Custom Precision Score: 0.9787 (When flagged positive, accuracy is 97.87%)
Custom Recall Score:    0.9583 (Captured 95.83% of all true positive cases)
sss
--- Performance Analysis at Threshold (0.35) ---
Custom Precision Score: 0.9490 (When flagged positive, accuracy is 94.90%)
Custom R

In [22]:
log_reg = LogisticRegression(
    penalty='l2',          # Use L2 (Ridge) regularization
    C=0.5,                 # Slightly stronger regularization than default
    solver='liblinear',        # Standard efficient solver
    max_iter=1000,         # Increased to guarantee optimization convergence
    random_state=42,
    class_weight='balanced'
)
threshold = 0.26

log_reg.fit(x_train, y_train)
test_probabilities = log_reg.predict_proba(x_test)[:, 1]
predictions = (test_probabilities >= threshold).astype(int)

print(classification_report(y_test, predictions))
print(confusion_matrix(y_test, predictions))

              precision    recall  f1-score   support

         0.0       1.00      0.87      0.93        30
         1.0       0.97      1.00      0.98       120

    accuracy                           0.97       150
   macro avg       0.98      0.93      0.96       150
weighted avg       0.97      0.97      0.97       150

[[ 26   4]
 [  0 120]]


In [19]:
import joblib

joblib.dump(log_reg, open("modelling/model/adult.jobllib", 'wb'))

In [20]:
np.savez("data/images/adult.npz", features=x_adult, labels=y_adult)

In [44]:
from xgboost import XGBClassifier